<a href="https://colab.research.google.com/github/ian-ren03/NLP-TC3009-A01384161/blob/main/notebooks/00_datos_protocolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, json, random, torch, numpy as np
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = SAVE_DIR = '/content/drive/MyDrive/PROYECTO FINAL NLP - ntm-en-es'
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo: ', device)
!pip install -q sacrebleu sentencepiece sacremoses

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo:  cuda


## Traer el repositorio

In [2]:
!git clone https://github.com/ian-ren03/NLP-TC3009-A01384161.git
%cd NLP-TC3009-A01384161
import sys; sys.path.append('.')

fatal: destination path 'NLP-TC3009-A01384161' already exists and is not an empty directory.
/content/NLP-TC3009-A01384161


## Protocolo obligatorio (Listing 1, sin modificar)


In [3]:
!wget -q https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip

import random, hashlib
with open('spa.txt', 'r', encoding='utf-8') as f:
    raw_pairs = [line.strip().split('\t')[:2] for line in f if '\t' in line]

filtered_pairs = [
    (en, es) for en, es in raw_pairs
    if 8 <= len(en.split()) <= 120 and len(es.split()) <= 120
]

rng = random.Random(23)
rng.shuffle(filtered_pairs)

n = len(filtered_pairs)
n_train = int(0.80 * n)
n_val = int(0.10 * n)
train_pairs = filtered_pairs[:n_train]
val_pairs = filtered_pairs[n_train:n_train + n_val]
test_pairs = filtered_pairs[n_train + n_val:]

test_hash = hashlib.sha256("\n".join([p[0] for p in test_pairs]).encode()).hexdigest()[:12]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print(f"SHA-256 Test Audit ID: {test_hash}")

Train: 29130 | Val: 3641 | Test: 3642
SHA-256 Test Audit ID: dbac04ac5648


## Candado de reproducibilidad

In [4]:
EXPECTED_HASH = 'dbac04ac5648'
assert test_hash == EXPECTED_HASH, 'La partición cambió: revisa filtro, semilla o archivo'

en_train = {p[0] for p in train_pairs}
en_test = {p[0] for p in test_pairs}
print('Oraciones inglesas repetidas en train_test: ', len(en_train & en_test))

Oraciones inglesas repetidas en train_test:  541


Hay 541 oraciones repetidas, esa es una limitante.

Vocabularios y pruebas

In [5]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.03 KiB | 350.00 KiB/s, done.
From https://github.com/ian-ren03/NLP-TC3009-A01384161
   58bf5f6..55fc5b1  main       -> origin/main
Updating 58bf5f6..55fc5b1
Fast-forward
 src/persist.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [6]:
!ls src

data.py  __init__.py  persist.py  __pycache__


In [7]:
from torch.utils.data import DataLoader
from src.data import *

eng_word2idx = build_vocab([en for en, _ in train_pairs], min_freq = 2, max_size = 20000)
spa_word2idx = build_vocab([es for _, es in train_pairs], min_freq = 2, max_size = 20000)
print('Vocab Inglés: ', len(eng_word2idx), '| Vocab Español:', len(spa_word2idx))

## Ratio de <unk>
def unk_rate(pairs, idx, col):
  toks = [idx.get(w, UNK_IDX) for p in pairs for w in clean_text(p[col]).split()]
  return sum(t == UNK_IDX for t in toks) / len(toks)

for name, ps in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]:
  print(f"{name}: unk EN = {unk_rate(ps, eng_word2idx, 0):.3%} | unk ES = {unk_rate(ps, spa_word2idx, 1):.3%}")

train_ds = NMTDataset(train_pairs, eng_word2idx, spa_word2idx)
sampler = BucketSampler([len(s) for s in train_ds.src], batch_size = 64)
train_loader = DataLoader(train_ds, batch_sampler = sampler, collate_fn = collate_fn)

src, src_lens, tgt = next(iter(train_loader))
print('src:', src.shape, ' | tgt:', tgt.shape, '| lens:', src_lens[:5])




Vocab Inglés:  5453 | Vocab Español: 8027
train: unk EN = 1.173% | unk ES = 2.708%
val: unk EN = 1.956% | unk ES = 4.393%
test: unk EN = 2.068% | unk ES = 4.365%
src: torch.Size([64, 9])  | tgt: torch.Size([64, 14]) | lens: tensor([9, 9, 9, 9, 9])


## Probar persistencia

In [8]:
from src.persist import save_experiment, load_experiment
m = torch.nn.Linear(2, 2).to(device)
opt = torch.optim.Adam(m.parameters())
save_experiment(m, opt, {'train_loss': [], 'val_loss': []}, 'prueba', SAVE_DIR)
m, h = load_experiment(torch.nn.Linear(2, 2), 'prueba', SAVE_DIR, device)
print('Recarga Ok:', h)

Checkpoint guardado en Drive: prueba.pt
Recarga Ok: {'train_loss': [], 'val_loss': []}
